In [1]:
import ollama

In [2]:
inventory_db={
    "laptop":{"stock":5,"base_price":1200},
    "monitor":{"stock":0,"base_price":300},
    "keyboard":{"stock":25,"base_price":80}
}

In [3]:
#DFINING TOOL
# TOOL 1:CHECKING DATABASE

def check_inventory(product_name):
    product_name=product_name.lower()

    if product_name in inventory_db:
        return inventory_db[product_name]

    return{"stock":0,"base_price":None}

In [4]:
#TOOL 2: BUSINESS LOGIC FOR DISCOUNTS
def calculate_loyalty_discount(base_price,years_as_customer):
    discount=min(years_as_customer * 0.05,0.30) #max 30%
    final_price=base_price*(1-discount)
    return round(final_price,2)

In [5]:
# MAP THE FUNCTIONS SO THE SCRIPT CAN CALL THEM BY NAME
available_functions={
    'check_inventory':check_inventory,
    'calculate_loyalty_discount':calculate_loyalty_discount,
}

In [6]:
#DEFINING SCEMA
tools=[
    {
        "type":"function",
        "function":{
            "name":"check_inventory",
            "description":"Get stock and price for a product",
            "parameters":{
                "type":"object",
                "properties":{
                    "product_name":{"type":"string"}
                },
                "required" :["product_name"]
            }

        }
    },
    {
        "type":"function",
        "function":{
            "name":"calculate_loyality_discount",
            "description":"Calculate final price based on loyalty years",
            "parameters":{
                "type":"object",
                "properties":{
                    "base_price":{"type":"number"},
                    "years_as_customer":{"type":"integer"}
                },
                "required":["base_price","years_as_customer"]
            }
        }
    }
]

In [7]:
tools

[{'type': 'function',
  'function': {'name': 'check_inventory',
   'description': 'Get stock and price for a product',
   'parameters': {'type': 'object',
    'properties': {'product_name': {'type': 'string'}},
    'required': ['product_name']}}},
 {'type': 'function',
  'function': {'name': 'calculate_loyality_discount',
   'description': 'Calculate final price based on loyalty years',
   'parameters': {'type': 'object',
    'properties': {'base_price': {'type': 'number'},
     'years_as_customer': {'type': 'integer'}},
    'required': ['base_price', 'years_as_customer']}}}]

In [8]:
#I am a customer of 5 years. What will be the final price of a laptop?
#I want to buy an iPhone.Can you check stock?"
message=[
    {"role":"user","content":"I am a customer of 5 years. What will be the final price of a laptop?"}
]

In [9]:
#CALLING LLM
response=ollama.chat(
    model="qwen3:8b",
    messages=message,
    tools=tools
)

In [10]:
response["message"]

Message(role='assistant', content='', thinking='Okay, let\'s see. The user has been a customer for 5 years and wants to know the final price of a laptop. They mentioned their loyalty years, so I need to check if there\'s a discount based on that.\n\nFirst, I should check the available functions. There\'s \'check_inventory\' to get stock and price, and \'calculate_loyality_discount\' to compute the final price with loyalty years. \n\nThe user is asking for the final price after applying the loyalty discount. So, I need the base price of the laptop. The \'check_inventory\' function can provide that. But wait, the user hasn\'t specified the laptop model. Hmm, maybe the function can still be called with the product name as \'laptop\'? Or maybe there\'s a default? The function\'s parameters require a product_name, so I should use that. \n\nWait, the user just said "laptop" without a specific model. Maybe the function can handle it by returning the base price for any laptop. Let me call \'ch

In [11]:
#check if the model asked to use any tools
tool_calls=response['message'].get('tool_calls')

if tool_calls:
    for tool_call in tool_calls:
        tool_name=tool_call['function']['name']
        tool_args=tool_call['function']['arguments']
        
        #Get the actual python function
        function_to_call=available_functions[tool_name]
        
        #Run the function with given arguments
        result=function_to_call(**tool_args)
        
        #Add the model's tool request to the conversation
        message.append({
            'role':'tool',
            'content':str(result)
        })
message

[{'role': 'user',
  'content': 'I am a customer of 5 years. What will be the final price of a laptop?'},
 {'role': 'tool', 'content': "{'stock': 5, 'base_price': 1200}"}]

In [12]:
#Ask the model to generate the final answer
final_response=ollama.chat(
    model='qwen3:8b',
    messages=message
)

print(final_response['message']['content'])

The base price of the laptop is $1200. However, the final price depends on factors like discounts, promotions, or loyalty benefits. Since you’ve been a customer for 5 years, you might qualify for a loyalty discount, but the exact amount isn’t specified here. If no additional details are provided, the **base price remains $1200**. For a precise final price, please share any applicable discounts or terms.
